# Image Feature Extraction Pipeline

This notebook extracts comprehensive image features for Amazon product listings:
- Deep embeddings (CNN, CLIP)
- Quality/composition metrics
- OCR features
- Optional object detection

Assumes CUDA T4 GPU and high RAM in Colab.


In [ ]:
# Install required packages (run once in Colab)
# !pip install torch torchvision timm open-clip-torch opencv-python-headless pytesseract easyocr ultralytics scikit-image scikit-learn pillow tqdm pandas pyarrow


In [ ]:
import os
import gc
import hashlib
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import cv2
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import pickle

# Deep learning
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import timm
import open_clip

# OCR
try:
    import pytesseract
    HAS_TESSERACT = True
except ImportError:
    HAS_TESSERACT = False
    print("Warning: pytesseract not available, OCR features will be skipped")

# Object detection (optional)
try:
    from ultralytics import YOLO
    HAS_YOLO = True
except ImportError:
    HAS_YOLO = False
    print("Warning: ultralytics not available, detection features will be skipped")

# Sklearn for PCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Image quality metrics
from skimage import filters, measure
from skimage.color import rgb2hsv

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


In [ ]:
# Configuration
IMG_DIR = Path("/images")  # Change to your images folder
CACHE_DIR = Path("./embedding_cache")
CACHE_DIR.mkdir(exist_ok=True)

# Load main dataset to get ASINs
MAIN_DF_PATH = Path("./../../data/final_datasets/40k_multisector_amazon_products.csv")
df_main = pd.read_csv(MAIN_DF_PATH)

print(f"Loaded {len(df_main)} products")
print(f"Images directory: {IMG_DIR}")
print(f"Images exist: {IMG_DIR.exists()}")


In [ ]:
# Map from images folder to dataset (reverse mapping approach)
# This avoids filesystem errors with long ASINs

# Get all unique ASINs from dataset as a set for fast lookup
all_asins = set(df_main['asin'].astype(str).unique())

# Create a mapping of hash -> ASIN for long ASINs (if they were hashed during download)
hash_to_asin = {}
for asin in all_asins:
    asin_str = str(asin).strip()
    if len(asin_str) > 200:  # Long ASINs were hashed
        hash_obj = hashlib.md5(asin_str.encode())
        hash_to_asin[hash_obj.hexdigest()] = asin_str

print(f"Total ASINs in dataset: {len(all_asins)}")
print(f"Long ASINs (will be hashed): {len(hash_to_asin)}")

# List all image files in the images folder
image_files = list(IMG_DIR.glob("*.jpg"))
print(f"Found {len(image_files)} image files in {IMG_DIR}")

# Map image files to ASINs
product_images = {}
unmatched_images = 0

for img_path in tqdm(image_files, desc="Mapping images to ASINs"):
    # Get filename without extension
    filename = img_path.stem  # This is the part without .jpg
    
    matched_asin = None
    
    # Try direct ASIN match first
    if filename in all_asins:
        matched_asin = filename
    # Try hash match (for long ASINs)
    elif filename in hash_to_asin:
        matched_asin = hash_to_asin[filename]
    # If filename looks like a hash (32 hex chars), try to match
    elif len(filename) == 32 and all(c in '0123456789abcdef' for c in filename.lower()):
        if filename in hash_to_asin:
            matched_asin = hash_to_asin[filename]
    
    if matched_asin:
        if matched_asin not in product_images:
            product_images[matched_asin] = []
        product_images[matched_asin].append(str(img_path))
    else:
        unmatched_images += 1

print(f"\nMapped {len(product_images)} products with images")
print(f"Total images mapped: {sum(len(imgs) for imgs in product_images.values())}")
print(f"Unmatched images (not in dataset): {unmatched_images}")

## Embedding Extractors


In [ ]:
class CNNEmbedder:
    """CNN embedding extractor using EfficientNet or ResNet"""
    def __init__(self, backbone="efficientnet_b0", device="cuda"):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.backbone = backbone
        
        # Load model
        self.model = timm.create_model(
            backbone,
            pretrained=True,
            num_classes=0,  # Remove classification head
            global_pool="avg"
        )
        self.model.eval()
        self.model.to(self.device)
        
        # Get embedding dimension
        with torch.no_grad():
            dummy = torch.randn(1, 3, 224, 224).to(self.device)
            self.embed_dim = self.model(dummy).shape[1]
        
        # Image preprocessing
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        print(f"CNN Embedder: {backbone}, dim={self.embed_dim}")
    
    def preprocess_image(self, img_path):
        """Load and preprocess image"""
        try:
            img = Image.open(img_path).convert('RGB')
            return self.transform(img)
        except Exception as e:
            return None
    
    def embed_batch(self, img_paths, batch_size=32):
        """Extract embeddings for a batch of images"""
        embeddings = []
        valid_paths = []
        
        for i in range(0, len(img_paths), batch_size):
            batch_paths = img_paths[i:i+batch_size]
            batch_tensors = []
            
            for path in batch_paths:
                tensor = self.preprocess_image(path)
                if tensor is not None:
                    batch_tensors.append(tensor)
                    valid_paths.append(path)
            
            if batch_tensors:
                batch = torch.stack(batch_tensors).to(self.device)
                with torch.no_grad():
                    emb = self.model(batch).cpu().numpy()
                    embeddings.append(emb)
        
        if embeddings:
            return np.vstack(embeddings), valid_paths
        return np.array([]), []

# Initialize CNN embedder
cnn_embedder = CNNEmbedder(backbone="efficientnet_b0", device="cuda")


In [ ]:
class CLIPEmbedder:
    """CLIP embedding extractor"""
    def __init__(self, model_name="ViT-B-32", device="cuda"):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.model_name = model_name
        
        # Load CLIP model
        self.model, _, self.preprocess = open_clip.create_model_and_transforms(
            model_name,
            pretrained="openai"
        )
        self.model.eval()
        self.model.to(self.device)
        
        # Get embedding dimension
        with torch.no_grad():
            dummy = self.preprocess(Image.new('RGB', (224, 224))).unsqueeze(0).to(self.device)
            self.embed_dim = self.model.encode_image(dummy).shape[1]
        
        print(f"CLIP Embedder: {model_name}, dim={self.embed_dim}")
    
    def preprocess_image(self, img_path):
        """Load and preprocess image"""
        try:
            img = Image.open(img_path).convert('RGB')
            return self.preprocess(img)
        except Exception as e:
            return None
    
    def embed_batch(self, img_paths, batch_size=64):
        """Extract embeddings for a batch of images"""
        embeddings = []
        valid_paths = []
        
        for i in range(0, len(img_paths), batch_size):
            batch_paths = img_paths[i:i+batch_size]
            batch_tensors = []
            
            for path in batch_paths:
                tensor = self.preprocess_image(path)
                if tensor is not None:
                    batch_tensors.append(tensor)
                    valid_paths.append(path)
            
            if batch_tensors:
                batch = torch.stack(batch_tensors).to(self.device)
                with torch.no_grad():
                    emb = self.model.encode_image(batch).cpu().numpy()
                    embeddings.append(emb)
        
        if embeddings:
            return np.vstack(embeddings), valid_paths
        return np.array([]), []

# Initialize CLIP embedder
clip_embedder = CLIPEmbedder(model_name="ViT-B-32", device="cuda")


## Quality & Composition Features


In [ ]:
def compute_quality_features(img_path):
    """Compute technical quality features for an image"""
    try:
        # Load image
        img = cv2.imread(str(img_path))
        if img is None:
            return None
        
        # Get file size
        filesize_kb = os.path.getsize(img_path) / 1024.0
        
        h, w = img.shape[:2]
        aspect_ratio = w / h if h > 0 else np.nan
        
        # Convert to different color spaces
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        
        # Brightness
        brightness_mean = float(gray.mean())
        brightness_std = float(gray.std())
        
        # Contrast (same as brightness_std)
        contrast = brightness_std
        
        # Saturation
        saturation_mean = float(hsv[:, :, 1].mean())
        saturation_std = float(hsv[:, :, 1].std())
        
        # Colorfulness (Hasler–Süsstrunk metric)
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        rg = rgb[:, :, 0].astype(np.float32) - rgb[:, :, 1].astype(np.float32)
        yb = (rgb[:, :, 0].astype(np.float32) + rgb[:, :, 1].astype(np.float32)) / 2 - rgb[:, :, 2].astype(np.float32)
        colorfulness = float(np.sqrt(np.mean(rg**2) + np.mean(yb**2)))
        
        # Sharpness (variance of Laplacian)
        sharpness_laplacian_var = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        
        # Blur score (inverse of sharpness)
        blur_score = 1.0 / (1.0 + sharpness_laplacian_var)
        
        # Entropy
        hist, _ = np.histogram(gray.flatten(), bins=256, range=(0, 256))
        hist = hist[hist > 0]  # Remove zeros
        hist = hist / hist.sum()  # Normalize
        entropy = float(-np.sum(hist * np.log2(hist + 1e-10)))
        
        # Exposure clipping
        exposure_clipped_high_pct = float(np.sum(gray > 250) / gray.size * 100)
        exposure_clipped_low_pct = float(np.sum(gray < 5) / gray.size * 100)
        
        return {
            'width': w,
            'height': h,
            'aspect_ratio': aspect_ratio,
            'filesize_kb': filesize_kb,
            'brightness_mean': brightness_mean,
            'brightness_std': brightness_std,
            'contrast': contrast,
            'saturation_mean': saturation_mean,
            'saturation_std': saturation_std,
            'colorfulness': colorfulness,
            'sharpness_laplacian_var': sharpness_laplacian_var,
            'blur_score': blur_score,
            'entropy': entropy,
            'exposure_clipped_high_pct': exposure_clipped_high_pct,
            'exposure_clipped_low_pct': exposure_clipped_low_pct
        }
    except Exception as e:
        return None


In [ ]:
def compute_composition_features(img_path):
    """Compute composition and background features"""
    try:
        img = cv2.imread(str(img_path))
        if img is None:
            return None
        
        h, w = img.shape[:2]
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # Edge density (Canny edges)
        edges = cv2.Canny(gray, 50, 150)
        edge_density = float(np.sum(edges > 0) / (w * h))
        
        # Background uniformity (std of border pixels)
        border_width = min(20, w // 10, h // 10)
        if border_width > 0:
            top_border = gray[:border_width, :].flatten()
            bottom_border = gray[-border_width:, :].flatten()
            left_border = gray[:, :border_width].flatten()
            right_border = gray[:, -border_width:].flatten()
            all_borders = np.concatenate([top_border, bottom_border, left_border, right_border])
            bg_uniformity = float(all_borders.std())
        else:
            bg_uniformity = np.nan
        
        # White background percentage
        white_threshold = 240
        white_pixels = np.sum((gray > white_threshold))
        white_bg_pct = float(white_pixels / (w * h) * 100)
        
        # Dominant color count (k-means on pixels, simplified)
        # Use a sample for efficiency
        sample_size = min(10000, w * h)
        indices = np.random.choice(w * h, sample_size, replace=False)
        pixels = img.reshape(-1, 3)[indices]
        
        from sklearn.cluster import KMeans
        try:
            kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
            kmeans.fit(pixels)
            dominant_color_count = len(np.unique(kmeans.labels_))
        except:
            dominant_color_count = 3
        
        # Saliency (simplified: use variance of gradients)
        grad_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        grad_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        saliency_map = np.sqrt(grad_x**2 + grad_y**2)
        
        saliency_mean = float(saliency_map.mean())
        saliency_max = float(saliency_map.max())
        saliency_peak_ratio = saliency_max / (saliency_mean + 1e-10)
        
        # Saliency center distance
        saliency_norm = saliency_map / (saliency_map.sum() + 1e-10)
        y_coords, x_coords = np.meshgrid(np.arange(h), np.arange(w), indexing='ij')
        center_y = float(np.sum(y_coords * saliency_norm))
        center_x = float(np.sum(x_coords * saliency_norm))
        img_center_y, img_center_x = h / 2, w / 2
        saliency_center_dist = float(np.sqrt((center_x - img_center_x)**2 + (center_y - img_center_y)**2))
        
        # Object occupancy proxy (salient area / image area)
        salient_threshold = saliency_mean * 2
        salient_area = np.sum(saliency_map > salient_threshold)
        object_occupancy_proxy = float(salient_area / (w * h))
        
        # Border clutter score (edges near borders)
        border_mask = np.zeros((h, w), dtype=bool)
        border_width_clutter = min(30, w // 10, h // 10)
        if border_width_clutter > 0:
            border_mask[:border_width_clutter, :] = True
            border_mask[-border_width_clutter:, :] = True
            border_mask[:, :border_width_clutter] = True
            border_mask[:, -border_width_clutter:] = True
            border_edges = np.sum(edges[border_mask] > 0)
            border_clutter_score = float(border_edges / (border_mask.sum() + 1e-10))
        else:
            border_clutter_score = np.nan
        
        return {
            'edge_density': edge_density,
            'bg_uniformity': bg_uniformity,
            'white_bg_pct': white_bg_pct,
            'dominant_color_count': dominant_color_count,
            'saliency_peak_ratio': saliency_peak_ratio,
            'saliency_center_dist': saliency_center_dist,
            'object_occupancy_proxy': object_occupancy_proxy,
            'border_clutter_score': border_clutter_score
        }
    except Exception as e:
        return None


## OCR Features


In [ ]:
def compute_ocr_features(img_path):
    """Compute OCR-based text features"""
    if not HAS_TESSERACT:
        return None
    
    try:
        img = cv2.imread(str(img_path))
        if img is None:
            return None
        
        # Run OCR
        try:
            ocr_text = pytesseract.image_to_string(img)
        except:
            ocr_text = ""
        
        # Basic counts
        ocr_char_count = len(ocr_text)
        words = ocr_text.split()
        ocr_word_count = len(words)
        
        # Numeric tokens
        import re
        numeric_pattern = re.compile(r'\d+')
        ocr_num_count = len(numeric_pattern.findall(ocr_text))
        
        # All caps ratio
        if ocr_word_count > 0:
            allcaps_words = sum(1 for w in words if w.isupper() and len(w) > 1)
            ocr_allcaps_ratio = allcaps_words / ocr_word_count
        else:
            ocr_allcaps_ratio = 0.0
        
        # Has units
        units_pattern = re.compile(r'\b(oz|lb|cm|mm|pack|count|pcs|pieces|ml|l|kg|g|inch|in|ft)\b', re.IGNORECASE)
        ocr_has_units = 1 if units_pattern.search(ocr_text) else 0
        
        # Has claims
        claims = [
            'bpa free', 'organic', 'non-toxic', 'premium', 'new',
            'natural', 'eco-friendly', 'biodegradable', 'recyclable',
            'certified', 'award', 'winner', 'best seller', 'top rated'
        ]
        ocr_has_claims = 0
        text_lower = ocr_text.lower()
        for claim in claims:
            if claim in text_lower:
                ocr_has_claims = 1
                break
        
        # Text overlay area proxy (simplified: use bounding boxes if available)
        try:
            ocr_data = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT)
            text_boxes = []
            for i, text in enumerate(ocr_data['text']):
                if text.strip():
                    x, y, w, h = ocr_data['left'][i], ocr_data['top'][i], ocr_data['width'][i], ocr_data['height'][i]
                    text_boxes.append((x, y, w, h))
            
            if text_boxes:
                total_box_area = sum(w * h for _, _, w, h in text_boxes)
                img_area = img.shape[0] * img.shape[1]
                text_overlay_area_proxy = total_box_area / img_area if img_area > 0 else 0.0
            else:
                text_overlay_area_proxy = 0.0
        except:
            text_overlay_area_proxy = 0.0
        
        return {
            'ocr_char_count': ocr_char_count,
            'ocr_word_count': ocr_word_count,
            'ocr_num_count': ocr_num_count,
            'ocr_allcaps_ratio': ocr_allcaps_ratio,
            'ocr_has_units': ocr_has_units,
            'ocr_has_claims': ocr_has_claims,
            'text_overlay_area_proxy': text_overlay_area_proxy
        }
    except Exception as e:
        return None


## Object Detection Features (Optional)


In [ ]:
# Initialize YOLO model if available
yolo_model = None
if HAS_YOLO:
    try:
        # Try to load local model first
        yolo_path = Path("./yolov8n.pt")
        if yolo_path.exists():
            yolo_model = YOLO(str(yolo_path))
        else:
            yolo_model = YOLO('yolov8n.pt')  # Will download
        print("YOLO model loaded")
    except Exception as e:
        print(f"Could not load YOLO: {e}")
        yolo_model = None
        HAS_YOLO = False

def compute_detector_features(img_path, model=None):
    """Compute object detection features (only on hero image or first few)"""
    if not HAS_YOLO or model is None:
        return None
    
    try:
        results = model(str(img_path), verbose=False)
        
        if len(results) == 0:
            return None
        
        result = results[0]
        boxes = result.boxes
        
        if boxes is None or len(boxes) == 0:
            return {
                'det_num_objects': 0,
                'det_main_box_area_ratio': 0.0,
                'det_main_box_center_dist': np.nan,
                'det_has_person': 0,
                'det_has_face': 0,
                'det_has_hand': 0,
                'det_secondary_objects_count': 0,
                'det_conf_mean': 0.0,
                'det_conf_max': 0.0
            }
        
        # Get detections
        confidences = boxes.conf.cpu().numpy()
        classes = boxes.cls.cpu().numpy()
        xyxy = boxes.xyxy.cpu().numpy()
        
        det_num_objects = len(boxes)
        det_conf_mean = float(confidences.mean())
        det_conf_max = float(confidences.max())
        
        # Main box (largest or highest confidence)
        main_idx = np.argmax(confidences)
        main_box = xyxy[main_idx]
        img_h, img_w = result.orig_shape
        
        main_box_area = (main_box[2] - main_box[0]) * (main_box[3] - main_box[1])
        img_area = img_w * img_h
        det_main_box_area_ratio = main_box_area / img_area if img_area > 0 else 0.0
        
        # Center distance
        box_center_x = (main_box[0] + main_box[2]) / 2
        box_center_y = (main_box[1] + main_box[3]) / 2
        img_center_x, img_center_y = img_w / 2, img_h / 2
        det_main_box_center_dist = float(np.sqrt((box_center_x - img_center_x)**2 + (box_center_y - img_center_y)**2))
        
        # Person/face/hand detection (COCO classes: person=0)
        # Note: YOLOv8n doesn't have separate face/hand classes, so we use person as proxy
        det_has_person = 1 if 0 in classes else 0
        det_has_face = 0  # Would need face detection model
        det_has_hand = 0  # Would need hand detection model
        
        # Secondary objects (excluding main)
        det_secondary_objects_count = det_num_objects - 1
        
        return {
            'det_num_objects': det_num_objects,
            'det_main_box_area_ratio': det_main_box_area_ratio,
            'det_main_box_center_dist': det_main_box_center_dist,
            'det_has_person': det_has_person,
            'det_has_face': det_has_face,
            'det_has_hand': det_has_hand,
            'det_secondary_objects_count': det_secondary_objects_count,
            'det_conf_mean': det_conf_mean,
            'det_conf_max': det_conf_max
        }
    except Exception as e:
        return None


## Main Processing Pipeline


In [ ]:
def process_single_image(img_path, asin, is_hero=False):
    """Process a single image and extract all features"""
    features = {'image_path': img_path, 'asin': asin, 'is_hero': is_hero}
    
    # Quality features
    quality = compute_quality_features(img_path)
    if quality:
        features.update(quality)
    
    # Composition features
    composition = compute_composition_features(img_path)
    if composition:
        features.update(composition)
    
    # OCR features
    ocr = compute_ocr_features(img_path)
    if ocr:
        features.update(ocr)
    
    # Detection features (only on hero or first few images)
    if is_hero and yolo_model is not None:
        det = compute_detector_features(img_path, yolo_model)
        if det:
            features.update(det)
    
    return features

# Process all images
print("Processing images...")
all_image_features = []

for asin, img_paths in tqdm(product_images.items(), desc="Products"):
    for idx, img_path in enumerate(img_paths):
        is_hero = (idx == 0)  # First image is hero
        features = process_single_image(img_path, asin, is_hero=is_hero)
        all_image_features.append(features)

print(f"Processed {len(all_image_features)} images")


In [ ]:
# Extract embeddings in batches
print("Extracting CNN embeddings...")
all_img_paths = [f['image_path'] for f in all_image_features]

cnn_embeddings_dict = {}
clip_embeddings_dict = {}

# CNN embeddings
cnn_embs, valid_paths_cnn = cnn_embedder.embed_batch(all_img_paths, batch_size=32)
for path, emb in zip(valid_paths_cnn, cnn_embs):
    cnn_embeddings_dict[path] = emb

print(f"Extracted {len(cnn_embeddings_dict)} CNN embeddings")

# CLIP embeddings
print("Extracting CLIP embeddings...")
clip_embs, valid_paths_clip = clip_embedder.embed_batch(all_img_paths, batch_size=64)
for path, emb in zip(valid_paths_clip, clip_embs):
    clip_embeddings_dict[path] = emb

print(f"Extracted {len(clip_embeddings_dict)} CLIP embeddings")

# Add embeddings to image features
for feat in all_image_features:
    path = feat['image_path']
    if path in cnn_embeddings_dict:
        for i, val in enumerate(cnn_embeddings_dict[path]):
            feat[f'cnn_emb_{i:04d}'] = float(val)
    if path in clip_embeddings_dict:
        for i, val in enumerate(clip_embeddings_dict[path]):
            feat[f'clip_emb_{i:04d}'] = float(val)


In [ ]:
def aggregate_product(image_feature_rows):
    """Aggregate image features per product"""
    if not image_feature_rows:
        return None
    
    # Separate hero and non-hero
    hero_row = next((r for r in image_feature_rows if r.get('is_hero', False)), image_feature_rows[0])
    all_rows = image_feature_rows
    
    product_feat = {'asin': hero_row.get('asin')}
    
    # Count
    product_feat['num_images'] = len(image_feature_rows)
    
    # Get all numeric columns (excluding metadata)
    exclude_cols = {'image_path', 'asin', 'is_hero'}
    numeric_cols = []
    for col in all_rows[0].keys():
        if col not in exclude_cols:
            try:
                # Check if numeric
                val = all_rows[0][col]
                if isinstance(val, (int, float)) and not isinstance(val, bool):
                    numeric_cols.append(col)
            except:
                pass
    
    # Aggregate: mean, max, std
    for col in numeric_cols:
        values = [r.get(col, np.nan) for r in all_rows if col in r]
        values = [v for v in values if not (isinstance(v, float) and np.isnan(v))]
        
        if values:
            product_feat[f'{col}_mean'] = float(np.mean(values))
            product_feat[f'{col}_max'] = float(np.max(values))
            product_feat[f'{col}_std'] = float(np.std(values)) if len(values) > 1 else 0.0
        
        # Hero version
        hero_val = hero_row.get(col, np.nan)
        if not (isinstance(hero_val, float) and np.isnan(hero_val)):
            product_feat[f'hero_{col}'] = float(hero_val)
    
    # Special: embedding statistics
    cnn_emb_cols = [col for col in numeric_cols if col.startswith('cnn_emb_')]
    clip_emb_cols = [col for col in numeric_cols if col.startswith('clip_emb_')]
    
    if cnn_emb_cols and clip_emb_cols:
        # Compute cosine similarity between CLIP and CNN embeddings
        cnn_embs_list = []
        clip_embs_list = []
        for row in all_rows:
            cnn_emb = np.array([row.get(col, 0) for col in cnn_emb_cols])
            clip_emb = np.array([row.get(col, 0) for col in clip_emb_cols])
            if cnn_emb.sum() != 0 and clip_emb.sum() != 0:
                cnn_embs_list.append(cnn_emb)
                clip_embs_list.append(clip_emb)
        
        if cnn_embs_list and clip_embs_list:
            # Average cosine similarity
            cos_sims = []
            for cnn_emb, clip_emb in zip(cnn_embs_list, clip_embs_list):
                # Normalize
                cnn_norm = cnn_emb / (np.linalg.norm(cnn_emb) + 1e-10)
                clip_norm = clip_emb / (np.linalg.norm(clip_emb) + 1e-10)
                # Pad to same length if needed
                min_len = min(len(cnn_norm), len(clip_norm))
                cos_sim = np.dot(cnn_norm[:min_len], clip_norm[:min_len])
                cos_sims.append(cos_sim)
            
            if cos_sims:
                product_feat['clip_cnn_cos_sim'] = float(np.mean(cos_sims))
        
        # Embedding variability
        if len(cnn_embs_list) > 1:
            cnn_stack = np.vstack(cnn_embs_list)
            product_feat['cnn_emb_intra_std'] = float(np.mean(np.std(cnn_stack, axis=0)))
        
        if len(clip_embs_list) > 1:
            clip_stack = np.vstack(clip_embs_list)
            product_feat['clip_emb_intra_std'] = float(np.mean(np.std(clip_stack, axis=0)))
    
    return product_feat

# Aggregate by product
print("Aggregating features by product...")
product_features = {}

for feat in all_image_features:
    asin = feat['asin']
    if asin not in product_features:
        product_features[asin] = []
    product_features[asin].append(feat)

aggregated_features = []
for asin, img_feats in tqdm(product_features.items(), desc="Aggregating"):
    agg = aggregate_product(img_feats)
    if agg:
        aggregated_features.append(agg)

print(f"Aggregated {len(aggregated_features)} products")


In [ ]:
# Optional: PCA compression for embeddings
print("Applying PCA compression...")

# Get embedding columns
cnn_emb_cols = [col for col in aggregated_features[0].keys() if col.startswith('cnn_emb_') and col.endswith('_mean')]
clip_emb_cols = [col for col in aggregated_features[0].keys() if col.startswith('clip_emb_') and col.endswith('_mean')]

# Extract base column names (remove _mean suffix)
cnn_base_cols = [col.replace('_mean', '') for col in cnn_emb_cols]
clip_base_cols = [col.replace('_mean', '') for col in clip_emb_cols]

if cnn_base_cols:
    # Build matrix
    cnn_matrix = []
    for feat in aggregated_features:
        row = [feat.get(f'{col}_mean', 0) for col in cnn_base_cols]
        cnn_matrix.append(row)
    cnn_matrix = np.array(cnn_matrix)
    
    # PCA
    pca_cnn = PCA(n_components=128)
    cnn_pca = pca_cnn.fit_transform(cnn_matrix)
    
    # Add to features
    for i, feat in enumerate(aggregated_features):
        for j in range(128):
            feat[f'cnn_pca_{j:04d}'] = float(cnn_pca[i, j])
    
    print(f"CNN PCA: {cnn_matrix.shape[1]} -> 128, explained variance: {pca_cnn.explained_variance_ratio_.sum():.3f}")

if clip_base_cols:
    # Build matrix
    clip_matrix = []
    for feat in aggregated_features:
        row = [feat.get(f'{col}_mean', 0) for col in clip_base_cols]
        clip_matrix.append(row)
    clip_matrix = np.array(clip_matrix)
    
    # PCA
    pca_clip = PCA(n_components=128)
    clip_pca = pca_clip.fit_transform(clip_matrix)
    
    # Add to features
    for i, feat in enumerate(aggregated_features):
        for j in range(128):
            feat[f'clip_pca_{j:04d}'] = float(clip_pca[i, j])
    
    print(f"CLIP PCA: {clip_matrix.shape[1]} -> 128, explained variance: {pca_clip.explained_variance_ratio_.sum():.3f}")


In [ ]:
# Convert to DataFrame and save
df_image_features = pd.DataFrame(aggregated_features)

print(f"Image features shape: {df_image_features.shape}")
print(f"Columns: {len(df_image_features.columns)}")

# Save as CSV
output_path = Path("./image_features_1.csv")
df_image_features.to_csv(output_path, index=False)
print(f"Saved image features to {output_path}")


In [ ]:
# Merge with main dataset
print("Merging with main dataset...")

# Ensure ASIN is string type for merging
df_main['asin'] = df_main['asin'].astype(str)
df_image_features['asin'] = df_image_features['asin'].astype(str)

# Merge
df_merged = df_main.merge(df_image_features, on='asin', how='left')

print(f"Merged dataset shape: {df_merged.shape}")
print(f"Products with image features: {df_merged['num_images'].notna().sum()}")

# Save merged dataset
merged_output_path = Path("./products_with_image_feats.csv")
df_merged.to_csv(merged_output_path, index=False)
print(f"Saved merged dataset to {merged_output_path}")
